In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from torch.utils.data import Dataset
import torch
import copy
from torch.optim.lr_scheduler import LinearLR
import os
from tqdm import tqdm
import numpy as np

/home/godwinkhalko/deformable-detr-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Let's start with a base Quen 2.5 0.5B instruct model
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", torch_dtype=torch.float16)
device = "cuda:1" if torch.cuda.is_available() else "cpu"
model.to(device)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

In [3]:
#Define a custom dataset class to handle the data. The data in this case is a simple arithmatic addition. 
# We'll randomly generate 2 numbers between 0-100 and generate the answer, in this case addition of the 2 numbers. 
# The prompt will be in the following format:
# You are a reasoning assistant.

# Solve the following problem.

# Respond in exactly this format.

# <think>
# Your reasoning
# </think>

# <answer>
# Final answer
# </answer>

# Question:

class ArthimaticAddition(Dataset):
    def __init__(self, num_samples=1000):
        self.num_samples = num_samples
        self.data = []
        for _ in range(num_samples):
            a = torch.randint(0, 100, (1,)).item()
            b = torch.randint(0, 100, (1,)).item()
            answer = a + b
            prompt = f"You are a reasoning assistant.\n\nSolve the following problem.\n\nRespond in exactly this format.\n\n<think>\nYour reasoning\n</think>\n\n<answer>\nFinal answer\n</answer>\n\nQuestion: What is {a} + {b}?"
            self.data.append((prompt, answer))

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.data[idx]

In [4]:
#Now we define a deterministic reward template function that will give a reward of 1 if the model's answer is in the correct format else 0. The correct format is as follows:
#<think>
#Your reasoning
#</think>
#<answer>
#Final answer
#</answer>

def reward_model(model_output, correct_answer, alpha=0.5):
    template_score = 0
    answer_score = 0
    # Check if the model output contains the correct format
    if "<think>" in model_output and "</think>" in model_output and "<answer>" in model_output and "</answer>" in model_output:
        #Ensure that the <think> and <answer> tags are in the correct order
        think_start = model_output.find("<think>")
        think_end = model_output.find("</think>")
        answer_start = model_output.find("<answer>")
        answer_end = model_output.find("</answer>")
        # print(think_start, think_end, answer_start, answer_end)
        #Make sure that the <think> and <answer> tags are in the correct order and output positvely starts with <think> and ends with </answer>
        if think_start == 0 and (think_start < think_end < answer_start < answer_end) and answer_end == len(model_output) - len("</answer>"):
            template_score = 1
        # Extract the answer from the model output
        model_answer = model_output[answer_start + len("<answer>"):answer_end].strip()
        
        # Check if the model's answer matches the correct answer
        if str(model_answer) == str(correct_answer):
            answer_score = 1  # Reward of 1 for correct format and correct answer
    return alpha * template_score + (1 - alpha) * answer_score  # Reward of 0 for incorrect format or incorrect answer

In [5]:
prompt = "You are a reasoning assistant.\n\nSolve the following problem.\n\nRespond in exactly this format.\n\n<think>\nYour reasoning\n</think>\n\n<answer>\nFinal answer\n</answer>\n\nQuestion: What is 23 + 45?"
model_old = model
num_return_sequences = 3  # Number of sequences to generate
# Tokenize the prompt
inputs = tokenizer(prompt, return_tensors="pt").to(device)
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

# Generate model output from the old model as per the number of return sequences specified
with torch.no_grad():
    outputs = model_old.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=200, num_return_sequences=num_return_sequences)


In [ ]:
def GRPO_Loop(model_old, model_ref, prompt, correct_answer, alpha=0.5, device="cuda:0", tokenizer=tokenizer, num_return_sequences=5):
    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    # Generate model output from the old model as per the number of return sequences specified
    with torch.no_grad():
        outputs = model_old.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=200, num_return_sequences=num_return_sequences)
    #Calculate the reward for each output based on the correct answer and the alpha value
    rewards = []
    model_outputs = []
    for output in outputs:
        model_output = tokenizer.decode(output[input_ids.shape[1]:], skip_special_tokens=True)
        reward = reward_model(model_output, correct_answer, alpha)
        rewards.append(reward)
        model_outputs.append(model_output)

    #Advantage calculation
    mean_reward = np.mean(rewards)
    std_reward = np.std(rewards)
    advantages = [(r - mean_reward) / (std_reward + 1e-8) for r in rewards]
    
    model_theta = GRPO_Update(model_old, model_ref, inputs, outputs, model_outputs, advantages, device=device)

    return model_theta, rewards



In [ ]:
def GRPO_Update(model_old, model_ref, inputs, outputs, model_outputs, advantages, device="cuda:0", tokenizer=tokenizer):
    # Create a copy of the old model to update
    model_theta = copy.deepcopy(model_old)
    model_theta.train()
    model_theta.to(device)

    # Concatenate the prompt and label tokens for the model input
    tokens = []
    model_theta_outputs = []
    model_ref_outputs = []
    model_old_outputs = [] 
    for i in range(len(model_outputs)):
        output = tokenizer(model_outputs[i], return_tensors="pt").to(device)
        tokens.append({
            "input_ids":      torch.cat([input["input_ids"],      output["input_ids"]],      dim=1),
            "attention_mask": torch.cat([input["attention_mask"],  output["attention_mask"]], dim=1),
        })
        answer_len = output["input_ids"].shape[1]
        with torch.no_grad():
            model_theta_output = model_theta(**tokens[i])
        model_ref_output = model_ref(**tokens[i])
        model_old_output = model_old(**tokens[i])
    
        model_theta_outputs.append(model_theta_output)
        model_ref_outputs.append(model_ref_output)
        model_old_outputs.append(model_old_output)

    prompt_len = inputs["input_ids"].shape[1]

    return model_theta
